In [15]:
from pathlib import Path
 
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

In [16]:
load_dotenv(find_dotenv(usecwd=True), override=True)

True

In [17]:
PROJECT_ROOT = Path.cwd().parent
DATA_FILE = PROJECT_ROOT / "data" / "2024_Microsoft.md"
INDEX_DIR = PROJECT_ROOT / "faiss_index"
print(DATA_FILE, DATA_FILE.exists())

d:\Project1\data\2024_Microsoft.md True


In [19]:
loader = TextLoader(str(DATA_FILE), encoding="utf-8-sig") 
documents = loader.load()
print("Total characters:", sum(len(d.page_content) for d in documents))

Total characters: 187530


In [22]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=80)
text_chunks = text_splitter.split_documents(documents)

In [23]:
text_chunks = [c for c in text_chunks if len(c.page_content.strip()) >= 100]
print(f"{len(text_chunks)} chunks after filtering")

342 chunks after filtering


In [24]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1907.80it/s]


In [25]:
if INDEX_DIR.exists():
    vectorstore = FAISS.load_local(
        str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True
    )
    print("Loaded existing index")
else:
    vectorstore = FAISS.from_documents(text_chunks, embeddings)
    vectorstore.save_local(str(INDEX_DIR))
    print("Built and saved new index")

Built and saved new index


In [26]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [27]:
query = "What was Microsoft's total revenue in fiscal year 2024?"
docs = vectorstore.similarity_search(query, k=4)
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
## _**Fiscal Year 2024 Compared with Fiscal Year 2023**_ 

Revenue increased $33.2 billion or 16% driven by growth across each of our segments. Intelligent Cloud revenue increased driven by Azure. Productivity and Business Processes revenue increased driven by Office 365 Commercial. More Personal Computing revenue increased driven by Gaming. 

Cost of revenue increased $8.3 billion or 13% driven by growth in Microsoft Cloud and Gaming, offset in part by a decline in Devices. 

30 

Gross margin increased $25.0 billion or 17% driven by growth across each of our segments.
--------------------------------------------------
Document 2:
## _**Fiscal Year 2024 Compared with Fiscal Year 2023**_ 

## _Productivity and Business Processes_ 

Revenue increased $8.5 billion or 12%. 

- Office Commercial products and cloud services revenue increased $5.8 billion or 14%. Office 365 Commercial revenue grew 16% with seat growth of 7%, driven by small and medium business and frontline worke

In [28]:
template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [29]:
prompt = ChatPromptTemplate.from_template(template)

In [34]:
llm_model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [35]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)
 
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | StrOutputParser()
)

In [36]:
print(rag_chain.invoke("What was Microsoft's total revenue in fiscal year 2024?"))

Microsoft’s total revenue for fiscal year 2024 was **$247.4 billion** (reported as $247,442 million).
